# Final Evaluation and Report Generation

## Deliverable 5 : Automated Dynamic Analysis Signature Generation

**Final Stage of the Research Pipeline**

This notebook synthesizes all artifacts from previous stages and generates comprehensive evaluation reports.

### Pipeline Stages:
1. ✓ `data_acquisition.ipynb` — Collect malware samples
2. ✓ `feature_extraction.ipynb` — Extract behavioral features
3. ✓ `behavior_ir.ipynb` — Build Behavior IR representation
4. ✓ `llm_behavior_analysis.ipynb` — LLM semantic analysis
5. ✓ `signature_generation_validation.ipynb` — Generate & validate signatures
6. → `final_evaluation_report.ipynb` — **CURRENT STAGE** — Comprehensive evaluation & reporting

### This Notebook:
- **Does NOT recompute** previous stages
- **Loads and aggregates** all pipeline artifacts
- **Computes evaluation metrics** (coverage, detection performance)
- **Generates visualizations** and statistical analysis
- **Produces final research tables** for publication
- **Exports per-sample analysis reports** for forensic review

---


---

## 🔴 IMPORTANT: Prerequisites Checklist

**This notebook MUST be run AFTER all previous stages!**

Before running this notebook, ensure you have executed these notebooks **IN THIS EXACT ORDER**:

### ✓ REQUIRED - Stage 1
- **Notebook:** `data_acquisition.ipynb`
- **Output:** Raw behavioral data (stored in sandbox databases)

### ✓ REQUIRED - Stage 2  
- **Notebook:** `feature_extraction.ipynb`
- **Output File:** `data/processed/behavior_features.csv` (must exist!)
- **Size:** ~8-10 MB

### ✓ REQUIRED - Stage 3
- **Notebook:** `behavior_ir.ipynb`
- **Output File:** 
  - `data/processed/behavior_details.json` (must exist!)
- **Size:** ~200-250 MB

### ✓ REQUIRED - Stage 4
- **Notebook:** `llm_behavior_analysis.ipynb`
- **Output:** Cached LLM analyses in `data/llm_analysis/cache/`

### ✓ REQUIRED - Stage 5
- **Notebook:** `signature_generation_validation.ipynb`
- **Output File:** `data/processed/behavioral_signatures.json` (must exist!)
- **Size:** ~40-50 MB

### ⭕ OPTIONAL - Ground Truth Labels
- **File:** `data/processed/public_labels.csv`
- **Purpose:** Testing accuracy metrics (if available)
- **Status:** Not required - notebook works without it

---

**ERROR in SECTION 2 means:** One or more required files (behavior_features.csv, behavior_details.json, behavioral_signatures.json) are missing from `data/processed/`

**SOLUTION:** Run the missing stages first, then come back to this notebook.

---


------------------------------------------------------------
## SECTION 1: Imports and Setup
------------------------------------------------------------


In [12]:
import json, os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# ── Paths ──────────────────────────────────────────────────
PROJECT_ROOT   = Path('../').resolve()
DATA_DIR       = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR    = PROJECT_ROOT / 'reports'
ANALYSIS_DIR   = REPORTS_DIR / 'generated_analysis'
EVAL_DIR       = REPORTS_DIR / 'evaluation'

for d in [REPORTS_DIR, ANALYSIS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Reports dir  : {REPORTS_DIR}")
print(f"Analysis dir : {ANALYSIS_DIR}")

# ── Timestamp for reports ──────────────────────────────
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"\nReport timestamp: {TIMESTAMP}")


Project root : C:\Users\MSI\Downloads\cyber_security\malware lab\Automated-Dynamic-Analysis-Signature-Generation
Data dir     : C:\Users\MSI\Downloads\cyber_security\malware lab\Automated-Dynamic-Analysis-Signature-Generation\data\processed
Reports dir  : C:\Users\MSI\Downloads\cyber_security\malware lab\Automated-Dynamic-Analysis-Signature-Generation\reports
Analysis dir : C:\Users\MSI\Downloads\cyber_security\malware lab\Automated-Dynamic-Analysis-Signature-Generation\reports\generated_analysis

Report timestamp: 20260316_175938


------------------------------------------------------------
## SECTION 2: Load Pipeline Artifacts
------------------------------------------------------------


In [13]:
# ── Check for required artifacts ────────────────────────
print("🔍 CHECKING REQUIRED FILES...")
print("=" * 70)
print(f"Looking in: {DATA_DIR}\n")

required_files = [
    'behavior_features.csv',
    'behavioral_signatures.json',
    'behavior_details.json'
]

missing_files = []
found_files = []

for fname in required_files:
    fpath = DATA_DIR / fname
    if not fpath.exists():
        missing_files.append(fname)
        print(f"✗ {fname:40s} NOT FOUND")
    else:
        size_mb = fpath.stat().st_size / (1024*1024)
        found_files.append((fname, size_mb))
        print(f"✓ {fname:40s} ({size_mb:.2f} MB)")

print()

if missing_files:
    print("🛑 ERROR: MISSING REQUIRED FILES")
    print("=" * 70)
    print(f"\nMissing {len(missing_files)} file(s):\n")
    for i, fname in enumerate(missing_files, 1):
        print(f"  {i}. {fname}")
    
    print("\n" + "-" * 70)
    print("📋 WHAT TO DO:")
    print("-" * 70)
    print("""
You MUST run the previous pipeline stages IN ORDER before running this notebook:

  Stage 1: data_acquisition.ipynb
           └─ Collects raw malware samples
           
  Stage 2: feature_extraction.ipynb
           └─ Must create: behavior_features.csv
           
  Stage 3: behavior_ir.ipynb
           └─ Must create: behavior_details.json
           
  Stage 4: llm_behavior_analysis.ipynb
           └─ Creates cached LLM analyses
           
  Stage 5: signature_generation_validation.ipynb
           └─ Must create: behavioral_signatures.json
           
  Stage 6: final_evaluation_report.ipynb
           └─ ← YOU ARE HERE (run this LAST)

🔗 EXECUTION ORDER:
   1 → 2 → 3 → 4 → 5 → 6

⚠️  DO NOT change the order!
    Each stage depends on outputs from previous stages.
    
📂 CURRENT STATUS:
   ✓ Found: {len(found_files)} file(s)
   ✗ Missing: {len(missing_files)} file(s)
   
💡 NEXT STEP:
   1. Check which files are missing (see list above)
   2. Find the corresponding notebook(s) that create those files
   3. Run that notebook(s) first
   4. Come back to this notebook
""")
    raise RuntimeError(
        f"\n❌ Missing {len(missing_files)} required file(s).\n"
        f"See the error message above for details on which notebooks to run.\n"
        f"Required files: {', '.join(missing_files)}"
    )

print("\n" + "=" * 70)
print("✓ ALL REQUIRED ARTIFACTS FOUND!")
print("=" * 70)
print(f"\n✓ Ready to proceed with evaluation ({len(found_files)}/{len(required_files)} files)\n")

🔍 CHECKING REQUIRED FILES...
Looking in: C:\Users\MSI\Downloads\cyber_security\malware lab\Automated-Dynamic-Analysis-Signature-Generation\data\processed

✓ behavior_features.csv                    (8.79 MB)
✓ behavioral_signatures.json               (45.45 MB)
✓ behavior_details.json                    (200.80 MB)


✓ ALL REQUIRED ARTIFACTS FOUND!

✓ Ready to proceed with evaluation (3/3 files)



In [14]:
# ── Load all artifacts ──────────────────────────────────
print("Loading artifacts...\n")

# Load features
features = pd.read_csv(DATA_DIR / 'behavior_features.csv')
print(f"Features:          {features.shape[0]:6d} samples, {features.shape[1]:4d} features")

# Load detailed behaviors (contains the IR data we need)
with open(DATA_DIR / 'behavior_details.json', 'r', encoding='utf-8') as f:
    behavior_ir = json.load(f)
print(f"Behavior details:  {len(behavior_ir):6d} samples")

# Load signatures
with open(DATA_DIR / 'behavioral_signatures.json', 'r', encoding='utf-8') as f:
    signatures = json.load(f)
print(f"Signatures:        {len(signatures):6d} samples")

# Load optional labels if available
labels_file = DATA_DIR / 'public_labels.csv'
if labels_file.exists():
    labels = pd.read_csv(labels_file)
    print(f"Ground truth:      {labels.shape[0]:6d} samples\n")
    HAS_LABELS = True
else:
    print(f"Ground truth:      NOT AVAILABLE (optional)\n")
    labels = None
    HAS_LABELS = False

print("✓ All artifacts loaded successfully!")


Loading artifacts...

Features:           48976 samples,   29 features
Behavior details:   48976 samples
Signatures:         48976 samples
Ground truth:      NOT AVAILABLE (optional)

✓ All artifacts loaded successfully!


------------------------------------------------------------
## SECTION 3: Dataset Statistics
------------------------------------------------------------


In [15]:
# ── Compute summary statistics from behavior data ────────
stats = {
    'total_samples': len(behavior_ir),
    'samples_with_behaviors': 0,
    'total_api_calls': 0,
    'samples_with_network': 0,
    'samples_with_registry': 0,
    'samples_with_file_io': 0,
    'samples_with_process_creation': 0,
    'avg_behaviors_per_sample': 0,
    'api_call_distribution': defaultdict(int),
    'behavioral_categories': Counter(),
}

for sample_hash, sample_data in behavior_ir.items():
    behaviors = sample_data.get('behaviors', [])
    if behaviors:
        stats['samples_with_behaviors'] += 1
    
    for behavior in behaviors:
        category = behavior.get('category', 'unknown')
        stats['behavioral_categories'][category] += 1
        
        if category == 'network':
            stats['samples_with_network'] += 1
        elif category == 'registry':
            stats['samples_with_registry'] += 1
        elif category == 'file':
            stats['samples_with_file_io'] += 1
        elif category == 'process':
            stats['samples_with_process_creation'] += 1
    
    api_calls = sample_data.get('api_calls', [])
    stats['total_api_calls'] += len(api_calls)
    stats['api_call_distribution'][len(api_calls)] += 1

if stats['samples_with_behaviors'] > 0:
    stats['avg_behaviors_per_sample'] = len(stats['behavioral_categories']) / stats['samples_with_behaviors']

# ── Display summary ────────────────────────────────────
print("\n" + "="*70)
print("DATASET SUMMARY STATISTICS")
print("="*70)
print()
print(f"Total Samples:                  {stats['total_samples']:,}")
print(f"Samples with Behaviors:         {stats['samples_with_behaviors']:,}")
print(f"Total API Calls:                {stats['total_api_calls']:,}")
print(f"Average API Calls/Sample:       {stats['total_api_calls']/max(1, stats['total_samples']):.1f}")
print()
print("Behavioral Categories:")
for cat, count in stats['behavioral_categories'].most_common():
    pct = 100.0 * count / max(1, len(stats['behavioral_categories']))
    print(f"  {cat:20s}: {count:6,d} ({pct:5.1f}%)")
print()
print(f"Samples with Network Activity:  {stats['samples_with_network']:,}")
print(f"Samples with Registry Changes: {stats['samples_with_registry']:,}")
print(f"Samples with File I/O:         {stats['samples_with_file_io']:,}")
print(f"Samples with Process Creation: {stats['samples_with_process_creation']:,}")
print("\n" + "="*70)


AttributeError: 'list' object has no attribute 'items'

------------------------------------------------------------
## SECTION 4: Signature Detection Evaluation
------------------------------------------------------------


In [ ]:
# ── Evaluate signature detection performance if labels exist ────
if HAS_LABELS:
    print("\n" + "="*70)
    print("SIGNATURE DETECTION EVALUATION")
    print("="*70 + "\n")
    
    # Create detection binary labels (1 = detected by signature, 0 = not detected)
    detected_samples = set(signatures.keys())
    y_true = []
    y_pred = []
    
    for idx, row in labels.iterrows():
        sample_hash = row.get('sample_hash') or row.get('hash') or row.get('sha256')
        if sample_hash:
            y_true.append(1)  # Assume all labeled samples are true positives
            y_pred.append(1 if sample_hash in detected_samples else 0)
    
    if y_true and y_pred:
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        accuracy = accuracy_score(y_true, y_pred)
        
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")
        print(f"F1-Score:   {f1:.4f}")
        print(f"Accuracy:   {accuracy:.4f}")
        print()
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        print("Confusion Matrix:")
        print(f"  TN: {cm[0,0]:6d}  FP: {cm[0,1]:6d}")
        print(f"  FN: {cm[1,0]:6d}  TP: {cm[1,1]:6d}")
        print()
        
        # Classification report
        report = classification_report(y_true, y_pred, output_dict=False)
        print("Classification Report:")
        print(report)
    else:
        print("⚠ Not enough labeled data for evaluation metrics.")
        print(f"  Available labels: {len(y_true)}")
else:
    print("\n⚠ Ground truth labels not available. Skipping performance evaluation.")
    print("   (This is optional - the pipeline works without labels.)")


------------------------------------------------------------
## SECTION 5: Signature Coverage Analysis
------------------------------------------------------------


In [ ]:
# ── Analyze signature coverage ──────────────────────────
print("\n" + "="*70)
print("SIGNATURE COVERAGE ANALYSIS")
print("="*70 + "\n")

total_samples = len(behavior_ir)
detected_samples = len(signatures)
coverage_pct = 100.0 * detected_samples / max(1, total_samples)

print(f"Total Samples:              {total_samples:,}")
print(f"Samples with Signatures:    {detected_samples:,}")
print(f"Coverage:                   {coverage_pct:.2f}%")
print()

# Analyze signature patterns
signature_types = Counter()
families = Counter()
mitre_techniques = Counter()

for sample_hash, sig_data in signatures.items():
    if isinstance(sig_data, dict):
        family = sig_data.get('family', 'unknown')
        families[family] += 1
        
        techniques = sig_data.get('mitre_techniques', [])
        for tech in techniques:
            mitre_techniques[tech] += 1

print(f"Unique Families:            {len(families)}")
print(f"Total MITRE Techniques:     {len(mitre_techniques)}")
print()

print("Top 10 Families:")
for family, count in families.most_common(10):
    pct = 100.0 * count / max(1, detected_samples)
    print(f"  {family:30s}: {count:4d} ({pct:5.1f}%)")

print()
print("Signature Statistics:")
print(f"  Total unique signatures:    {detected_samples}")
print(f"  Unique families:            {len(families)}")
print(f"  Avg samples/family:         {detected_samples/max(1, len(families)):.1f}")


------------------------------------------------------------
## SECTION 6: MITRE ATT&CK Analysis
------------------------------------------------------------


In [ ]:
# ── Analyze MITRE ATT&CK techniques ────────────────────
print("\n" + "="*70)
print("MITRE ATT&CK TECHNIQUE ANALYSIS")
print("="*70 + "\n")

# Collect all MITRE techniques
all_techniques = Counter()
technique_by_sample = defaultdict(set)

for sample_hash, sample_data in behavior_ir.items():
    behaviors = sample_data.get('behaviors', [])
    for behavior in behaviors:
        techniques = behavior.get('mitre', [])
        for tech in techniques:
            all_techniques[tech] += 1
            technique_by_sample[sample_hash].add(tech)

print(f"Total Unique Techniques Detected: {len(all_techniques)}\n")

print("Top 20 Most Frequent Techniques:")
for tech, count in all_techniques.most_common(20):
    pct = 100.0 * count / max(1, len(behavior_ir))
    print(f"  {tech:50s}: {count:5,d} ({pct:5.1f}%)")

# Technique distribution statistics
tech_counts = list(all_techniques.values())
print(f"\nTechnique Frequency Statistics:")
print(f"  Mean:      {np.mean(tech_counts):.1f}")
print(f"  Median:    {np.median(tech_counts):.1f}")
print(f"  Std Dev:   {np.std(tech_counts):.1f}")
print(f"  Min:       {np.min(tech_counts)}")
print(f"  Max:       {np.max(tech_counts)}")


------------------------------------------------------------
## SECTION 7: Behavioral Pattern Analysis
------------------------------------------------------------


In [ ]:
# ── Analyze high-level behavioral patterns ──────────────
print("\n" + "="*70)
print("BEHAVIORAL PATTERN ANALYSIS")
print("="*70 + "\n")

# Define behavior pattern keywords
patterns = {
    'Process Injection': ['CreateRemoteThread', 'WriteProcessMemory', 'NtCreateThreadEx'],
    'Persistence': ['SetValueEx', 'RegCreateKey', 'WriteFile', 'CreateProcess'],
    'C2 Communication': ['InternetConnect', 'HttpSendRequest', 'socket', 'connect',
                       'WSASocket', 'recv', 'send'],
    'File Dropping': ['WriteFile', 'CreateFile', 'CopyFile', 'MoveFile'],
    'Privilege Escalation': ['ImpersonateLoggedOnUser', 'OpenProcessToken', 'DuplicateToken'],
    'Defense Evasion': ['SetWindowsHookEx', 'NtSetInformationFile', 'RegDeleteKey'],
}

pattern_stats = {pattern: {'count': 0, 'samples': set()} for pattern in patterns}

# Scan all behaviors for patterns
for sample_hash, sample_data in behavior_ir.items():
    behaviors = sample_data.get('behaviors', [])
    for behavior in behaviors:
        api_name = behavior.get('api', '').lower()
        for pattern, keywords in patterns.items():
            for keyword in keywords:
                if keyword.lower() in api_name:
                    pattern_stats[pattern]['count'] += 1
                    pattern_stats[pattern]['samples'].add(sample_hash)
                    break

# Display results
print(f"{'Pattern':<25} {'Samples':<15} {'Coverage':<15} {'Total Events'}")
print("-" * 70)
for pattern, stats_dict in sorted(pattern_stats.items(),
                                   key=lambda x: len(x[1]['samples']),
                                   reverse=True):
    sample_count = len(stats_dict['samples'])
    coverage = 100.0 * sample_count / max(1, len(behavior_ir))
    event_count = stats_dict['count']
    print(f"{pattern:<25} {sample_count:>6,d}       {coverage:>6.1f}%        {event_count:>8,d}")


------------------------------------------------------------
## SECTION 8: Visualization
------------------------------------------------------------


In [ ]:
# ── Behavioral Category Distribution ──────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Behavioral Category Distribution
categories = dict(stats['behavioral_categories'].most_common(10))
ax = axes[0, 0]
ax.barh(list(categories.keys()), list(categories.values()), color='steelblue')
ax.set_xlabel('Frequency', fontsize=11)
ax.set_title('Top 10 Behavioral Categories', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Plot 2: Top MITRE Techniques
top_techniques = dict(all_techniques.most_common(10))
ax = axes[0, 1]
ax.barh(list(top_techniques.keys()), list(top_techniques.values()), color='coral')
ax.set_xlabel('Frequency', fontsize=11)
ax.set_title('Top 10 MITRE ATT&CK Techniques', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Plot 3: Signature Coverage
coverage_data = [detected_samples, total_samples - detected_samples]
labels_pie = [f'Signatures\n{detected_samples:,}\n({coverage_pct:.1f}%)',
              f'No Signature\n{total_samples - detected_samples:,}\n({100-coverage_pct:.1f}%)']
ax = axes[1, 0]
ax.pie(coverage_data, labels=labels_pie, autopct='%1.1f%%',
       colors=['#2ecc71', '#e74c3c'], startangle=90)
ax.set_title('Signature Generation Coverage', fontsize=12, fontweight='bold')

# Plot 4: Top Malware Families
top_families = dict(families.most_common(8))
ax = axes[1, 1]
ax.bar(range(len(top_families)), list(top_families.values()), color='mediumpurple')
ax.set_xticks(range(len(top_families)))
ax.set_xticklabels(list(top_families.keys()), rotation=45, ha='right')
ax.set_ylabel('Sample Count', fontsize=11)
ax.set_title('Top 8 Malware Families', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(EVAL_DIR / f'analysis_overview_{TIMESTAMP}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved visualization: {EVAL_DIR / f'analysis_overview_{TIMESTAMP}.png'}")


In [ ]:
# ── Behavioral Pattern Heatmap ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for pattern analysis
pattern_names = list(patterns.keys())
pattern_coverage = []
pattern_events = []

for pattern in pattern_names:
    sample_count = len(pattern_stats[pattern]['samples'])
    event_count = pattern_stats[pattern]['count']
    pattern_coverage.append(100.0 * sample_count / max(1, len(behavior_ir)))
    pattern_events.append(event_count)

# Create bar chart
x = np.arange(len(pattern_names))
width = 0.35

bars1 = ax.bar(x - width/2, pattern_coverage, width, label='Sample Coverage (%)',
               color='skyblue', alpha=0.8)
ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, pattern_events, width, label='Total Events',
                color='lightcoral', alpha=0.8)

ax.set_xlabel('Behavioral Pattern', fontsize=11)
ax.set_ylabel('Sample Coverage (%)', fontsize=11, color='skyblue')
ax2.set_ylabel('Total Events', fontsize=11, color='lightcoral')
ax.set_title('Behavioral Pattern Frequency and Coverage', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(pattern_names, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

fig.legend(loc='upper right', bbox_to_anchor=(0.99, 0.99))
plt.tight_layout()
plt.savefig(EVAL_DIR / f'behavioral_patterns_{TIMESTAMP}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization: {EVAL_DIR / f'behavioral_patterns_{TIMESTAMP}.png'}")


------------------------------------------------------------
## SECTION 9: Automatic Malware Report Generation
------------------------------------------------------------


In [ ]:
# ── Generate per-sample analysis reports ────────────────
def generate_sample_report(sample_hash: str) -> dict:
    """Generate structured analysis report for a single sample."""
    report = {
        'sample_hash': sample_hash,
        'threat_level': 'Unknown',
        'key_behaviors': [],
        'network_indicators': [],
        'file_indicators': [],
        'process_indicators': [],
        'registry_indicators': [],
        'mitre_techniques': [],
        'signature': '',
        'confidence_score': 0.0,
    }
    
    # Get IR data
    ir = behavior_ir.get(sample_hash, {})
    behaviors = ir.get('behaviors', [])
    
    # Collect indicators and techniques
    threat_score = 0
    for behavior in behaviors:
        api = behavior.get('api', '')
        category = behavior.get('category', '')
        args = behavior.get('args', {})
        
        report['key_behaviors'].append({
            'api': api,
            'category': category,
            'context': behavior.get('context', '')
        })
        
        if category == 'network':
            report['network_indicators'].append({
                'api': api,
                'target': args.get('target', 'unknown')
            })
            threat_score += 2
        elif category == 'file':
            report['file_indicators'].append({
                'api': api,
                'path': args.get('path', 'unknown')
            })
            threat_score += 1
        elif category == 'process':
            report['process_indicators'].append({
                'api': api,
                'process': args.get('process', 'unknown')
            })
            threat_score += 1
        elif category == 'registry':
            report['registry_indicators'].append({
                'api': api,
                'key': args.get('key', 'unknown')
            })
            threat_score += 1
        
        # Collect MITRE techniques
        techniques = behavior.get('mitre', [])
        report['mitre_techniques'].extend(techniques)
    
    # Determine threat level
    if threat_score >= 10:
        report['threat_level'] = 'CRITICAL'
        report['confidence_score'] = 0.95
    elif threat_score >= 5:
        report['threat_level'] = 'HIGH'
        report['confidence_score'] = 0.85
    elif threat_score >= 2:
        report['threat_level'] = 'MEDIUM'
        report['confidence_score'] = 0.70
    else:
        report['threat_level'] = 'LOW'
        report['confidence_score'] = 0.50
    
    # Get signature if available
    sig = signatures.get(sample_hash, {})
    if isinstance(sig, dict):
        report['signature'] = sig.get('signature_code', '')
        report['family'] = sig.get('family', 'unknown')
    
    # Remove duplicates from MITRE techniques
    report['mitre_techniques'] = list(set(report['mitre_techniques']))
    
    return report

print("Generating per-sample analysis reports...\n")

# Generate reports for all samples
sample_reports = {}
threat_distribution = Counter()

for sample_hash in behavior_ir.keys():
    report = generate_sample_report(sample_hash)
    sample_reports[sample_hash] = report
    threat_distribution[report['threat_level']] += 1

print(f"✓ Generated reports for {len(sample_reports)} samples\n")

print("Threat Level Distribution:")
for level in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
    count = threat_distribution.get(level, 0)
    pct = 100.0 * count / max(1, len(sample_reports))
    print(f"  {level:10s}: {count:5,d} ({pct:5.1f}%)")


In [ ]:
# ── Export individual sample reports ────────────────────
print("\nExporting individual sample reports...\n")

export_count = 0
for sample_hash, report in sample_reports.items():
    # Export as JSON
    report_json = json.dumps(report, indent=2, ensure_ascii=False, default=str)
    report_path = ANALYSIS_DIR / f"{sample_hash}.json"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report_json)
    export_count += 1
    
    if export_count % 100 == 0:
        print(f"  [{export_count:5d}] Exported sample reports...")

print(f"\n✓ Exported {export_count} individual sample reports")
print(f"  Location: {ANALYSIS_DIR}")


------------------------------------------------------------
## SECTION 10: Final Research Tables
------------------------------------------------------------


In [ ]:
# ── Generate summary tables for research publication ────
print("\n" + "="*70)
print("FINAL RESEARCH TABLES")
print("="*70 + "\n")

# Table 1: Dataset Statistics
print("Table 1: Dataset Statistics")
print("-" * 70)
table1 = pd.DataFrame([
    ['Total Samples', stats['total_samples']],
    ['Samples with Behaviors', stats['samples_with_behaviors']],
    ['Total API Calls', stats['total_api_calls']],
    ['Avg API Calls/Sample', f"{stats['total_api_calls']/max(1, stats['total_samples']):.1f}"],
    ['Samples with Network Activity', stats['samples_with_network']],
    ['Samples with Registry Changes', stats['samples_with_registry']],
    ['Samples with File I/O', stats['samples_with_file_io']],
    ['Samples with Process Creation', stats['samples_with_process_creation']],
], columns=['Metric', 'Value'])
print(table1.to_string(index=False))

# Table 2: Signature Coverage
print("\n\nTable 2: Signature Generation Coverage")
print("-" * 70)
table2 = pd.DataFrame([
    ['Total Samples', total_samples],
    ['Samples with Signatures', detected_samples],
    ['Coverage (%)', f"{coverage_pct:.2f}%"],
    ['Unique Families', len(families)],
    ['Unique MITRE Techniques', len(all_techniques)],
    ['Avg Samples per Family', f"{detected_samples/max(1, len(families)):.1f}"],
], columns=['Metric', 'Value'])
print(table2.to_string(index=False))

# Table 3: Top Malware Families
print("\n\nTable 3: Top 15 Malware Families")
print("-" * 70)
table3_data = []
for rank, (family, count) in enumerate(families.most_common(15), 1):
    pct = 100.0 * count / max(1, detected_samples)
    table3_data.append([rank, family, count, f"{pct:.1f}%"])
table3 = pd.DataFrame(table3_data, columns=['Rank', 'Family', 'Samples', '%'])
print(table3.to_string(index=False))

# Table 4: Top MITRE Techniques
print("\n\nTable 4: Top 20 MITRE ATT&CK Techniques")
print("-" * 70)
table4_data = []
for rank, (tech, count) in enumerate(all_techniques.most_common(20), 1):
    pct = 100.0 * count / max(1, len(behavior_ir))
    table4_data.append([rank, tech, count, f"{pct:.1f}%"])
table4 = pd.DataFrame(table4_data, columns=['Rank', 'Technique', 'Samples', '%'])
print(table4.to_string(index=False))


In [ ]:
# ── Export summary tables as CSV ────────────────────────
print("\n\nExporting research tables...\n")

# Export dataset statistics
table1.to_csv(EVAL_DIR / f'table1_dataset_statistics_{TIMESTAMP}.csv', index=False)
print(f"✓ Table 1: {EVAL_DIR / f'table1_dataset_statistics_{TIMESTAMP}.csv'}")

# Export signature coverage
table2.to_csv(EVAL_DIR / f'table2_signature_coverage_{TIMESTAMP}.csv', index=False)
print(f"✓ Table 2: {EVAL_DIR / f'table2_signature_coverage_{TIMESTAMP}.csv'}")

# Export top families
table3.to_csv(EVAL_DIR / f'table3_top_families_{TIMESTAMP}.csv', index=False)
print(f"✓ Table 3: {EVAL_DIR / f'table3_top_families_{TIMESTAMP}.csv'}")

# Export top techniques
table4.to_csv(EVAL_DIR / f'table4_top_techniques_{TIMESTAMP}.csv', index=False)
print(f"✓ Table 4: {EVAL_DIR / f'table4_top_techniques_{TIMESTAMP}.csv'}")

print(f"\n✓ All research tables exported to: {EVAL_DIR}")


In [ ]:
# ── Final Summary ──────────────────────────────────────
print("\n" + "="*70)
print("PIPELINE EXECUTION SUMMARY")
print("="*70 + "\n")

print("✓ STAGE 1: data_acquisition.ipynb")
print(f"  └─ Collected {total_samples:,} malware samples\n")

print("✓ STAGE 2: feature_extraction.ipynb")
print(f"  └─ Extracted {features.shape[1]} behavioral features\n")

print("✓ STAGE 3: behavior_ir.ipynb")
print(f"  └─ Built Behavior IR for {len(behavior_ir):,} samples\n")

print("✓ STAGE 4: llm_behavior_analysis.ipynb")
print(f"  └─ Generated semantic analyses (cached for reuse)\n")

print("✓ STAGE 5: signature_generation_validation.ipynb")
print(f"  └─ Generated {detected_samples:,} CAPEv2 signatures ({coverage_pct:.1f}% coverage)\n")

print("✓ STAGE 6: final_evaluation_report.ipynb (CURRENT)")
print(f"  └─ Evaluation metrics computed")
print(f"  └─ {len(all_techniques)} MITRE ATT&CK techniques identified")
print(f"  └─ {len(families)} malware families characterized")
print(f"  └─ {export_count:,} individual sample reports generated")
print(f"  └─ Research tables exported for publication")
print(f"  └─ Visualizations saved\n")

print("Output Locations:")
print(f"  Individual Reports:  {ANALYSIS_DIR}")
print(f"  Research Tables:     {EVAL_DIR}")
print(f"  Visualizations:      {EVAL_DIR}")

print(f"\n{'='*70}")
print("✓ Complete pipeline executed successfully!")
print(f"{'='*70}")
